In [1]:
import re
import random
import html
import urllib.parse
from collections import Counter
from typing import List

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score

# ---------------- CONFIG ----------------
DATA_PATH = "data/XSS_dataset.csv"
MAX_VOCAB = 20000
MAX_LEN = 150
BATCH_SIZE = 64
EPOCHS = 30
LR = 1e-3
PATIENCE = 5
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "best_xss_model.pth"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [2]:
def xss_preprocess(text: str) -> str:
    if pd.isna(text):
        return ""

    s = str(text)

    # normalize
    s = s.replace("\x00", "")
    s = urllib.parse.unquote(s)
    s = html.unescape(s)

    # unicode escape
    s = re.sub(r"\\u([0-9a-fA-F]{4})",
               lambda m: chr(int(m.group(1), 16)), s)

    s = s.lower()

    # separate html symbols
    s = s.replace("<", " < ").replace(">", " > ")
    s = s.replace("=", " = ").replace("(", " ( ").replace(")", " ) ")

    s = re.sub(r"\s+", " ", s).strip()
    return s


In [4]:
def xss_tokenize(text: str) -> List[str]:
    pattern = re.compile(
        r"</?\w+|"          # html tags
        r"\w+|"             # words
        r"on\w+|"           # event handlers
        r"javascript:|data:|"
        r"[<>=\"'();:/.\-]|"
        r"\S"
    )
    return pattern.findall(text)[:MAX_LEN]


In [5]:
df = pd.read_csv(DATA_PATH)
texts = df["sentence"].astype(str).values
labels = df["Label"].astype(int).values

x_train, x_tmp, y_train, y_tmp = train_test_split(
    texts, labels, test_size=0.4, stratify=labels, random_state=SEED
)

x_val, x_test, y_val, y_test = train_test_split(
    x_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=SEED
)

train_tokens = [xss_tokenize(xss_preprocess(t)) for t in x_train]
val_tokens   = [xss_tokenize(xss_preprocess(t)) for t in x_val]
# test_tokens  = [xss_tokenize(xss_preprocess(t)) for t in x_test]


In [6]:
counter = Counter()
for t in train_tokens:
    counter.update(t)

vocab = ["<PAD>", "<UNK>"] + [w for w, _ in counter.most_common(MAX_VOCAB)]
word2idx = {w: i for i, w in enumerate(vocab)}


In [7]:
def encode(tokens_list):
    out = []
    for tokens in tokens_list:
        seq = [word2idx.get(t, 1) for t in tokens][:MAX_LEN]
        seq += [0] * (MAX_LEN - len(seq))
        out.append(seq)
    return out

train_seqs = encode(train_tokens)
val_seqs   = encode(val_tokens)
# test_seqs  = encode(test_tokens)


In [8]:
class XSSDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self): return len(self.y)

    def __getitem__(self, i):
        return torch.LongTensor(self.x[i]), torch.tensor(self.y[i])

counts = np.bincount(y_train)
weights = 1. / torch.tensor(counts, dtype=torch.float)
sampler = WeightedRandomSampler(weights[y_train], len(y_train))

train_loader = DataLoader(XSSDataset(train_seqs, y_train),
                          batch_size=BATCH_SIZE, sampler=sampler)
val_loader   = DataLoader(XSSDataset(val_seqs, y_val),
                          batch_size=BATCH_SIZE)
# test_loader  = DataLoader(XSSDataset(test_seqs, y_test),
#                           batch_size=BATCH_SIZE)


In [9]:
class Attention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc = nn.Linear(dim, 1, bias=False)

    def forward(self, x, mask):
        scores = self.fc(x).squeeze(-1)
        scores = scores.masked_fill(mask == 0, -1e9)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(x * weights.unsqueeze(-1), dim=1)
        return context


In [10]:
class XSSDetector(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.emb = nn.Embedding(vocab_size, 128, padding_idx=0)

        self.convs = nn.ModuleList([
            nn.Conv1d(128, 128, k, padding=k//2) for k in [3,5,7]
        ])

        self.lstm = nn.LSTM(384, 256,
                            bidirectional=True,
                            batch_first=True)

        self.attn = Attention(512)

        self.fc = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        mask = (x != 0).float()

        x = self.emb(x).permute(0,2,1)
        x = torch.cat([F.relu(c(x)) for c in self.convs], dim=1)
        x = x.permute(0,2,1)

        x, _ = self.lstm(x)

        att = self.attn(x, mask)
        mx  = torch.max(x, dim=1)[0]

        out = torch.cat([att, mx], dim=1)
        return self.fc(out)


In [11]:
model = XSSDetector(len(vocab)).to(DEVICE)

criterion = nn.CrossEntropyLoss(
    weight=weights.to(DEVICE),
    label_smoothing=0.05
)

optimizer = optim.AdamW(model.parameters(), lr=LR)

best_auc = 0
pat = 0

for epoch in range(EPOCHS):
    model.train()
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        probs = torch.softmax(
            model(torch.LongTensor(val_seqs).to(DEVICE)), dim=1
        )[:,1].cpu().numpy()

    auc = roc_auc_score(y_val, probs)
    print(f"Epoch {epoch+1} | Val AUC: {auc:.5f}")

    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), MODEL_PATH)
        pat = 0
    else:
        pat += 1
        if pat >= PATIENCE:
            break


Epoch 1 | Val AUC: 0.99995
Epoch 2 | Val AUC: 0.99998
Epoch 3 | Val AUC: 0.99999
Epoch 4 | Val AUC: 0.99999
Epoch 5 | Val AUC: 0.99999
Epoch 6 | Val AUC: 0.99999
Epoch 7 | Val AUC: 0.99999
Epoch 8 | Val AUC: 0.99999


In [12]:
model.load_state_dict(torch.load(MODEL_PATH))
model.eval()
test_tokens  = [xss_tokenize(xss_preprocess(t)) for t in x_test]
test_seqs  = encode(test_tokens)
test_loader  = DataLoader(XSSDataset(test_seqs, y_test),
                          batch_size=BATCH_SIZE)

with torch.no_grad():
    probs = torch.softmax(
        model(torch.LongTensor(test_seqs).to(DEVICE)), dim=1
    )[:,1].cpu().numpy()

print("AUC:", roc_auc_score(y_test, probs))
print("ACC:", accuracy_score(y_test, probs > 0.5))


AUC: 0.9999946320973738
ACC: 0.9985390796201608


In [13]:
def test_on_xss_payload_list_txt(txt_path: str = 'xss-payload-list.txt',
                                  batch_size: int = 64):
    """
    تست مدل روی لیست معروف XSS payloadها (فایل txt)
    همه payloadها malicious فرض می‌شوند
    """
    print(f"\n=== تست مدل روی لیست XSS payloadها ({txt_path}) ===")

    # خواندن مستقیم txt
    with open(txt_path, 'r', encoding='utf-8', errors='ignore') as f:
        payloads = [line.strip().strip('"').strip("'") for line in f
                    if line.strip() and not line.startswith('#')]

    total_samples = len(payloads)
    print(f"تعداد payloadهای لود شده: {total_samples:,}")

    # همه لیبل‌ها Malicious (1)
    labels = np.array([1] * total_samples)

    print("نمونه چند payload اول:")
    for i in range(min(5, total_samples)):
        print(f"   {i+1}: {payloads[i][:100]}...")

    # پیش‌پردازش و توکنایز
    print("\nدر حال پیش‌پردازش و توکنایز...")
    clean_texts = [xss_preprocess(p) for p in payloads]
    tokens_list = [xss_tokenize(t) for t in clean_texts]
    sequences = encode(tokens_list)

    # پیش‌بینی باتچی
    print(f"\nدر حال پیش‌بینی با باتچ‌های {batch_size} تایی...")
    model.eval()
    probs_list = []

    with torch.no_grad():
        for i in range(0, len(sequences), batch_size):
            batch = torch.LongTensor(sequences[i:i + batch_size]).to(DEVICE)
            batch_probs = torch.softmax(model(batch), dim=1)[:, 1].cpu().numpy()
            probs_list.append(batch_probs)

            if (i // batch_size + 1) % 50 == 0 or i + batch_size >= len(sequences):
                processed = min(i + batch_size, len(sequences))
                print(f"   پردازش شده: {processed:,}/{total_samples:,}")

    probs = np.concatenate(probs_list)
    preds = (probs > 0.5).astype(int)

    # محاسبه معیارها (فقط روی کلاس malicious معنی‌دار است)
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

    acc = accuracy_score(labels, preds)
    recall = recall_score(labels, preds)  # مهم‌ترین: چند درصد payloadها رو گرفته؟
    precision = precision_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    auc = roc_auc_score(labels, probs)

    print("\n" + "="*60)
    print("نتایج روی لیست XSS payloadها (همه Malicious):")
    print("="*60)
    print(f"تعداد payloadها: {total_samples:,}")
    print(f"Accuracy:  {acc:.5f}  (چون همه malicious، برابر با Recall است)")
    print(f"Recall:    {recall:.5f}  ← مهم‌ترین معیار اینجا!")
    print(f"Precision: {precision:.5f}")
    print(f"F1-Score:  {f1:.5f}")
    print(f"AUC:       {auc:.5f}")
    print("="*60)

    # تعداد payloadهایی که از دست رفته (false negative)
    missed = np.sum((preds == 0) & (labels == 1))
    print(f"تعداد payloadهایی که مدل تشخیص نداد (Missed): {missed:,} از {total_samples:,}")

    if missed > 0:
        print("\nچند payload که مدل اشتباه تشخیص داد (Benign گفت):")
        missed_indices = np.where((preds == 0) & (labels == 1))[0]
        for idx in missed_indices[:10]:  # حداکثر ۱۰ تا
            print(f"   احتمال: {probs[idx]:.4f} | Payload: {payloads[idx][:100]}...")

    return {
        'num_payloads': total_samples,
        'missed': missed,
        'recall': recall,
        'auc': auc,
        'probs': probs,
        'preds': preds
    }

In [14]:
results = test_on_xss_payload_list_txt('data/xss-payload-list.txt')


=== تست مدل روی لیست XSS payloadها (data/xss-payload-list.txt) ===
تعداد payloadهای لود شده: 6,613
نمونه چند payload اول:
   1: -prompt(8)-...
   2: -prompt(8)-...
   3: ;a=prompt,a()//...
   4: ;a=prompt,a()//...
   5: -eval("window['pro'%2B'mpt'](8)")-...

در حال پیش‌پردازش و توکنایز...

در حال پیش‌بینی با باتچ‌های 64 تایی...
   پردازش شده: 3,200/6,613
   پردازش شده: 6,400/6,613
   پردازش شده: 6,613/6,613

نتایج روی لیست XSS payloadها (همه Malicious):
تعداد payloadها: 6,613
Accuracy:  0.97807  (چون همه malicious، برابر با Recall است)
Recall:    0.97807  ← مهم‌ترین معیار اینجا!
Precision: 1.00000
F1-Score:  0.98892
AUC:       nan
تعداد payloadهایی که مدل تشخیص نداد (Missed): 145 از 6,613

چند payload که مدل اشتباه تشخیص داد (Benign گفت):
   احتمال: 0.1637 | Payload: -prompt(8)-...
   احتمال: 0.1637 | Payload: -prompt(8)-...
   احتمال: 0.0682 | Payload: -eval("window['pro'%2B'mpt'](8)")-...
   احتمال: 0.0682 | Payload: -eval("window['pro'%2B'mpt'](8)")-...
   احتمال: 0.4918 | Payload:

C:\Users\BARCODE\Desktop\JupyterProject\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


In [15]:
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

def test_on_new_dataset(
    csv_path: str,
    model,
    word2idx: dict,
    text_col: str = "sentence",
    label_col: str = "Label",
    has_label: bool = True,
    threshold: float = 0.5,
    batch_size: int = 128
):
    df = pd.read_csv(csv_path)

    texts = df[text_col].astype(str).values
    labels = df[label_col].values if has_label else None

    # preprocess + tokenize
    tokens = [xss_tokenize(xss_preprocess(t)) for t in texts]
    seqs = encode(tokens)

    model.eval()
    probs = []

    with torch.no_grad():
        for i in range(0, len(seqs), batch_size):
            batch = torch.LongTensor(seqs[i:i+batch_size]).to(DEVICE)
            p = torch.softmax(model(batch), dim=1)[:, 1]
            probs.extend(p.cpu().numpy())

    probs = np.array(probs)
    preds = (probs >= threshold).astype(int)

    print("📊 Prediction summary")
    print(pd.Series(preds).value_counts())

    if has_label:
        print("\n✅ Evaluation metrics")
        print("AUC:", roc_auc_score(labels, probs))
        print("Accuracy:", accuracy_score(labels, preds))
        print("\nConfusion Matrix:")
        print(confusion_matrix(labels, preds))
        print("\nClassification Report:")
        print(classification_report(labels, preds, digits=4))

    # append results
    df["xss_prob"] = probs
    df["xss_pred"] = preds

    return df


In [16]:
model.load_state_dict(torch.load("best_xss_model.pth"))
model.to(DEVICE)

df_results = test_on_new_dataset(
    csv_path="data/xss_da.csv",
    model=model,
    word2idx=word2idx,
    has_label=True
)

df_results.to_csv("xss_test_results.csv", index=False)


📊 Prediction summary
0    33331
1     8415
Name: count, dtype: int64

✅ Evaluation metrics
AUC: 0.9996896644466101
Accuracy: 0.9746562544914483

Confusion Matrix:
[[33323  1050]
 [    8  7365]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9998    0.9695    0.9844     34373
           1     0.8752    0.9989    0.9330      7373

    accuracy                         0.9747     41746
   macro avg     0.9375    0.9842    0.9587     41746
weighted avg     0.9778    0.9747    0.9753     41746

